# Instalando Xee e inicializando GEE

In [ ]:
# importando o GEE e Geemap
import ee
import geemap

# inicializando GEE
geemap.ee_initialize(project='ee-enrique', opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
# instalando Xee
!pip install -q xee

# importa bibliotecas
import xee
import xarray as xr

# Definindo região: `Itajubá`

In [ ]:
# seleciona região
itajuba = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

# criando um mapa interativo
Map = geemap.Map()

# centra o mapa na região
Map.centerObject(itajuba, zoom=12)

# adicionando basemap
Map.add_basemap('SATELLITE')

# contorno da região
style = {'color': 'yellow', 'fillColor': '00000000'}
Map.addLayer(itajuba.style(**style), {}, 'itajuba')

# exibe na tela
Map

# Definindo região: `Lago`

In [ ]:
# seleciona região
itajuba = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

# criando um mapa interativo
Map = geemap.Map()

# centra o mapa na região
Map.centerObject(itajuba, zoom=12)

# adicionando basemap
Map.add_basemap('SATELLITE')

# contorno da região
style = {'color': 'yellow', 'fillColor': '00000000'}
Map.addLayer(itajuba.style(**style), {}, 'itajuba')

# exibe na tela
Map

In [ ]:
#  estrai as coordenadas da região selecionada
lago = Map.draw_last_feature.geometry()

In [ ]:
lago

In [ ]:
# região de interesse
regiao = ee.Geometry.Rectangle([-45.4400, -22.4132, -45.4359, 22.4088])

In [ ]:
# lago
lago = ee.Geometry.Rectangle([-45.4400, -22.4132, -45.4359, -22.4088])

# criando um mapa interativo
Map = geemap.Map()

# centraliza o mapa
Map.centerObject(lago, 17)

# adicionando basemap
Map.add_basemap('SATELLITE')

# contorno da região
style = {'color': 'gray', 'fillColor': '00000000'}
Map.addLayer(lago, style, 'Região')

# exibe na tela
Map

# Processamento

In [ ]:
# lago
lago = ee.Geometry.Rectangle([-45.4400, -22.4132, -45.4359, -22.4088])

# criando uma função que calcula o NDTI
def turbidez(img):

    # seleciona imagens com menos de 20% de nuvens
    cloud = img.select('probability')
    cloud_free = cloud.lt(20)

    # aplica o fator de escala do das bandas de reflectância
    sr = img.select('B.*').multiply(0.0001)

    # calcula o NDWI e seleciona valores acima de 0.1
    ndwi = sr.normalizedDifference(['B3', 'B8']).rename('ndwi')
    water_body = ndwi.gt(0.1)

    # calcula o NDTI
    ndti = sr.normalizedDifference(['B4', 'B3']).rename('ndti')

    return ndti.updateMask(cloud_free).updateMask(water_body).copyProperties(img, ['system:time_start'])

# carregando os dados
sen2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").linkCollection(ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY"), 'probability') \
         .filterBounds(lago) \
         .filterDate('2020', '2025') \
         .map(turbidez)
sen2

In [ ]:
# usando 100 m de resolução espacial
ds = xr.open_dataset(sen2,
                     engine = 'ee',
                     crs = 'EPSG:4326',
                     scale = 0.001,
                     geometry = lago)

# essa multiplicação por "1" aumenta a velocidade de processamento. Não muda nada nos dados
ds = ds.sortby('time') * 1

# mostra os dados
ds

In [ ]:
# reamostrando para escala mensal
ds_mensal = ds.resample(time = 'M').median('time')
ds_mensal

In [ ]:
# gráfico. Plotando todas as imagens da coleção. Lembre-se são imagens do Sentinel-2 de 2024, a cada 5 dias.
ds_mensal.ndti.plot(x = 'lon',
                    y = 'lat',
                    col = 'time',
                    col_wrap = 12,
                    robust = True,
                    cmap ='coolwarm')

In [ ]:
# calcula a média espacial
ds_mean = ds_mensal.mean(dim = ['lat', 'lon']).to_dataframe()

In [ ]:
# plota grafico linear
ds_mean.plot()